# Gemma 4 Dev Agent: What Moves the Leaderboard (updated 9 Oct)

**The short version.** The public top went from 0.12 to 0.24 in two weeks. The bundles that got there did *less*, not more: one agent, no graph tools, a hard tool-call budget, and a prompt that forces an edit early. The first version of this guide recommended the opposite on three points. This version says which ones and shows the evidence.

**What changed since the first version**

| First version said | Evidence now says | Source |
|---|---|---|
| Analyzer + coder split is high impact | +1 task out of 129 in a controlled run, inside noise | Ayush Thakur's ablations (discussion 746250) |
| Use graph tools first | All 21 semantic searches in a 129-task run returned nothing; the best public bundles do not register them | same thread; budget-fit notebook |
| Custom Python is a hard block | Python *tools* and callbacks are blocked, but a skill can ship `.py` scripts the agent runs with `run_skill_script` | yiyu0716's v19 bundle (public 0.17) |
| Budget is 60 min / 500 turns | That is the harness default. Your `eval_config.yaml` sets the real per-task cap, and the 12-hour total is what bites | competition `HARNESS_README.md`, staff thread 747519 |

Everything below separates what I measured, what someone else published, and what is still a guess.


## 1. Read the harness spec from the competition data


In [ ]:
# Read the harness spec straight from the attached competition data, so this
# notebook stays honest to the source rather than to my memory of it.
import glob

def find_readme():
    for p in glob.glob("/kaggle/input/**/HARNESS_README.md", recursive=True):
        return p
    return None

readme = find_readme()
if readme:
    text = open(readme, encoding="utf-8").read()
    print(f"Found harness spec: {readme}  ({len(text):,} chars)\n")
    for line in text.splitlines():
        if line.startswith(("## ", "### ")):
            print(line)
else:
    text = ""
    print("HARNESS_README.md not attached. Add the competition as an input "
          "(Add Input -> the competition) to run the grounded cells.")


## 2. How one task is scored

Each task is a real issue in a frozen repo. The agent edits `/workspace`; the harness turns the working tree into a `git diff`; a fresh checkout gets that diff plus hidden tests. **All hidden tests pass, or the task scores zero.** There is no partial credit, and an empty diff is an automatic zero.

```mermaid
flowchart LR
  I[Issue text] --> A[Single Gemma 4 agent]
  R[Repo at base_commit] --> A
  A -->|edit by call ~12| D[git diff]
  D --> G[Fresh tree + hidden tests]
  G --> S[Pass or fail]
```

About 120 hidden tasks, split evenly between the public and private leaderboards. **One public task is worth about 0.017**, so 0.13 to 0.18 is three tasks.


## 3. The 12-hour budget is the real constraint

All hidden tasks have to finish inside 12 hours on 4×L4, or the whole submission errors. Your `eval_config.yaml` sets the per-task limits:

```yaml
evaluation:
  timeout_seconds: 240   # per command, and also bounds the grader's pytest run
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
```

Oleksii Zhukov reported that a 4.5-minute cap took about 10 hours in a scored run, so each task carries roughly half a minute of setup and verification on top of the cap. The cell below turns that into a worst-case check. It assumes every task runs to the cap, which is pessimistic: a tool-call cap makes many tasks stop early.


In [ ]:
def worst_case_hours(cap_min, n_tasks=120, overhead_min=0.5, fixed_setup_min=15):
    """Hours if every task runs to the time cap. overhead_min is per-task setup + verification."""
    return (n_tasks * (cap_min + overhead_min) + fixed_setup_min) / 60

print(f"{'cap (min)':>9} | {'worst case (h)':>14} | fits 12h?")
for cap in (4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0):
    h = worst_case_hours(cap)
    print(f"{cap:>9} | {h:>14.1f} | {'yes' if h <= 12 else 'only if many tasks stop early'}")


The budget-fit bundle (public 0.18) uses 8 minutes with a 28-call cap, and its author reports the scored run finished in time because most tasks stop at the call cap or submit early. That is a bet on the hidden tasks, not a guarantee. If you raise the call cap, lower the time cap.


## 4. Which settings matter: a controlled ablation

Ayush Thakur ran the same 129 local tasks under one change at a time (discussion 746250, credit to him; numbers copied from his post). These are single runs on faster GPUs than Kaggle's, so treat a 2–3 task gap as noise. The large gaps are not noise.


In [ ]:
import matplotlib.pyplot as plt

# Resolved / 129, from Ayush Thakur's post (discussion 746250). E9 is the reference config;
# the summaries-off run changed the temperature 1.0 config (51), not E9.
runs = [
    ("Reference (temp 0.2, thinking 4096)", 48),
    ("Temperature 0.0", 47),
    ("Temperature 0.4", 51),
    ("Temperature 1.0", 51),
    ("Top-p 0.80", 48),
    ("Top-p 1.00", 48),
    ("Thinking budget 2048", 50),
    ("Thinking budget 6144", 48),
    ("Summaries off (base: temp 1.0 = 51)", 35),
    ("Thinking disabled", 23),
]
labels, vals = zip(*runs)
INK, MUTED, BLUE, SURFACE = "#0b0b0b", "#52514e", "#2a78d6", "#fcfcfb"

fig, ax = plt.subplots(figsize=(8, 4.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
y = range(len(runs))[::-1]
ax.barh(list(y), vals, color=BLUE, height=0.6)
ax.axvline(48, color=MUTED, lw=1, ls="--")
ax.text(48.5, len(runs) - 0.4, "reference", color=MUTED, fontsize=9, va="bottom")
for yi, v in zip(y, vals):
    ax.text(v + 0.6, yi, str(v), va="center", fontsize=9, color=INK)
ax.set_yticks(list(y), labels, fontsize=9, color=INK)
ax.set_xlabel("Tasks resolved out of 129", color=MUTED)
ax.set_xlim(0, 60)
ax.set_title("Turning thinking or its summaries off is the only large effect",
             loc="left", fontsize=11, color=INK)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.spines["left"].set_color(MUTED); ax.spines["bottom"].set_color(MUTED)
ax.tick_params(colors=MUTED)
plt.tight_layout(); plt.show()


**What to take from it**

- Keep thinking on **and** keep `include_thoughts: true`. Disabling thinking halved the resolved count. Hiding the summaries cost 16 tasks against its own base (the temperature 1.0 run, 51).
- Temperature, top-p and thinking budget moved results by 0–3 tasks, which is inside the noise of one run.
- In the same thread: cutting the tool budget from 50 to 25 calls cost 14 tasks, and adding an analyzer sub-agent gained 1.


## 5. How noisy is one public score?

My own submission log, unedited. Same harness and same model throughout.

| Date | Bundle | Public |
|---|---|---|
| 25 Sep | baseline + an extra navigation skill | 0.00 |
| 26 Sep | clean analyzer + coder baseline | 0.08 |
| 27 Sep | coder holds the graph tools directly | 0.08 |
| 1 Oct | baseline at temperature 0.7 | 0.06 |
| 2 Oct | unmodified fork of the public 0.12 bundle | 0.13 |

The 26 Sep and 2 Oct bundles are nearly the same and scored 0.08 and 0.13. That is about three tasks of run-to-run variation. The cell puts a number on it.


In [ ]:
import math

def binomial_ci(p, n, z=1.96):
    se = math.sqrt(p * (1 - p) / n)
    return p - z * se, p + z * se

n_public = 60  # about half of ~120 hidden tasks
for p in (0.12, 0.18, 0.24):
    lo, hi = binomial_ci(p, n_public)
    print(f"true rate {p:.2f}: one public score lands in about [{lo:.2f}, {hi:.2f}] 95% of the time")


With about 60 public tasks, a single score has a 95% range of roughly ±0.08–0.11 around the true rate. Two bundles 0.03 apart are not distinguishable on one submission each, and picking your luckiest public draw tends to regress on the private half.


## 6. What the stronger public bundles share

| Bundle | Public | Shape |
|---|---|---|
| romanrozen, "LB TOP 1" | 0.12 | analyzer + coder, graph tools, no `eval_config` budget |
| yiyu0716, v19 | 0.17 | analyzer + editor, temp 1.0, 7.5 min / 100 calls, `g4-edit` SEARCH/REPLACE skill |
| verracodeguacas, Budget-Fit Single Agent | 0.18 | single agent, 8 min / 28 calls / 240 s, prompt says edit by call 12, graph tools unused |
| honghanhhh, "LB 0.24" | 0.24 as stated in the title | budget-fit settings + graph tools unregistered + contract-first prompt |

Scores are as the authors state them. I have not reproduced them. The pattern they share:

1. **One agent.** The analyzer adds context and turns without a measured gain.
2. **A hard tool-call cap plus an early-edit rule.** Runs that are still only reading after about 12 calls rarely succeed.
3. **Copy the contract from the issue.** The hidden tests check exact names, messages and status codes, so a paraphrased error string fails.
4. **Protect the diff.** Scratch files go in `/tmp`, test files are never touched, and the agent checks `git diff` before `submit_patch`.
5. **Submit once an edit exists.** A red local test is not the grader, and waiting for green burns the clock.


## 7. Validate before you spend a daily submission

Scored submissions are limited to about one a day, and since 8 Oct a team can only have **one pending** at a time (staff thread 747519). Queues have run 20+ hours. A malformed bundle costs a day. The cell below is a dependency-free structural check; Kaggle's published wheelhouse gives you the real compiler if you want the full check.


In [ ]:
# Dependency-free structural validator — mirrors the host's hard checks.
import re, pathlib

ALLOWED = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
ROOTS = ["agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml"]

def validate(root):
    root = pathlib.Path(root)
    if not root.exists():
        print(f"(no dir at {root} — validator is reusable; point it at your bundle)"); return False
    problems = []
    roots = [n for n in ROOTS if (root / n).exists()]
    if len(roots) != 1: problems.append(f"need exactly one root config, found {roots}")
    models, includes = set(), []
    for y in root.rglob("*"):
        if y.suffix in (".yaml", ".yml"):
            txt = y.read_text(encoding="utf-8", errors="ignore")
            models |= set(re.findall(r"^\s*model:\s*['\"]?([^'\"\s#]+)", txt, re.M))
            for ref in re.findall(r"!include\s+(\S+)", txt) + re.findall(r"config_path:\s*(\S+)", txt):
                if not (y.parent / ref).resolve().exists(): includes.append(f"{y.name} -> {ref}")
    if includes: problems.append(f"unresolved include/config_path: {includes}")
    if len(models) != 1: problems.append(f"must declare exactly one base model, found {sorted(models)}")
    files = [p for p in root.rglob("*") if p.is_file()]
    bad_ext = [str(p.relative_to(root)) for p in files if p.suffix.lower() not in ALLOWED]
    if bad_ext: problems.append(f"disallowed extensions: {bad_ext}")
    total = sum(p.stat().st_size for p in files)
    if total > 3 * 1024**3: problems.append(f"bundle too big: {total} bytes")
    if problems:
        print("BLOCKING:"); [print("  -", p) for p in problems]; return False
    print(f"OK: 1 root, single model {sorted(models)}, {len(files)} files, {total} bytes"); return True


## 8. A single-agent bundle you can build on

This writes the budget-fit configuration (credit: verracodeguacas) with a short prompt of my own that encodes the five points from section 6. It validates the bundle and writes `submission.zip`. It is a starting point, not a tuned submission. I have not scored this exact prompt.


In [ ]:
import pathlib, shutil, zipfile

BUNDLE = pathlib.Path("/kaggle/working/submission") if pathlib.Path("/kaggle/working").is_dir() \
    else pathlib.Path("submission")
shutil.rmtree(BUNDLE, ignore_errors=True)

FILES = {
"agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
generate_content_config: !include configs/sampling.yaml
""",
"configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
"eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
""",
"prompts/system.md": """You fix one issue in the repository at /workspace with the smallest source change.
Hidden tests are applied to your git diff afterwards. An empty diff scores zero.

Budget: 28 tool calls, 8 minutes. When either runs out, the working tree is graded as it is.

1. Before any tool call, list the exact names, messages, status codes and return values the issue requires. Use them character for character.
2. Locate with one `git grep -n "<most specific string>" -- '*.py' | head -20`, then read at most 120 lines around the hit.
3. By your 12th tool call a source file must be edited. If unsure, edit the most likely function now.
4. Run at most one focused check (`pytest path::test -q` or `python -c ...`). Never run bare `pytest`.
   Fix only assertion failures about your change. Ignore fixture, import or pre-existing failures.
5. Scratch files go in /tmp. Never create or edit tests, conftest.py or pytest.ini.
6. Run `git diff`, revert any test path with `git checkout --`, then call submit_patch once.
""",
}
for name, text in FILES.items():
    p = BUNDLE / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")

if validate(BUNDLE):
    out = BUNDLE.parent / "submission.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(BUNDLE.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(BUNDLE))
    print("wrote", out, sorted(zipfile.ZipFile(out).namelist()))


## 9. LoRA, briefly

Adapters are allowed and scored through vLLM `--lora-modules`. Ayush's view after his sweeps is that the remaining gains are in post-training, which fits the ablation: prompt and sampling changes move a few tasks at most. Two cautions from the discussion: the official sample submission's adapters failed to score early on, and whether you may distil trajectories from external LLMs was still unanswered by the organizers in thread 742807 when I last read it. Smoke-test a tiny adapter end to end before investing days.


## 10. Bottom line

- Start from a single agent with an `eval_config.yaml` budget, not the analyzer + coder baseline.
- Keep thinking and thought summaries on. Do not spend submissions on temperature.
- Budget for 12 hours across about 120 tasks before you raise any cap.
- One public score is ±0.08 or so. Compare bundles on several runs or a local eval, not on one leaderboard line.

Corrections are welcome in the comments. If someone has task-level results showing the analyzer or graph tools helping, I would like to see them, and I will update this guide.

*Sources: the competition `HARNESS_README.md`; discussions 746250 (Ayush Thakur), 747519 (Kaggle staff), 742807; public notebooks by verracodeguacas, honghanhhh, yiyu0716 and romanrozen.*


---
### 🤝 Feedback & Discussion
- Author: **Hitarth Jain**
- Companion Silver Kernel: [Gemma 4 Patch Berserker: Evaluate, Then Submit](https://www.kaggle.com/code/hitarthjain0/gemma-4-patch-berserker-evaluate-then-submit)
- *If this guide helped you avoid a wasted daily submission or break through the 0.12 barrier, please leave an **upvote**! Feedback and questions welcome in the comments below.*